# Notebook 09 — Final Model Validation, Statistical Robustness, and Research Consolidation

**Project:** ECIPS — leakage-resistant classification of Kaggle-provided transaction risk labels.

## Research objectives

1. Re-evaluate ExtraTrees, LightGBM, and Logistic Regression using the same chronological validation and test partitions.
2. Compare both multiclass classification and high-risk one-vs-rest screening objectives.
3. Assess probability calibration and threshold sensitivity.
4. Estimate uncertainty with bootstrap confidence intervals, while acknowledging that ordinary row bootstrap does not fully model temporal dependence.
5. Evaluate multiple forward-chaining temporal windows.
6. Consolidate findings and limitations from the Notebook 07 and Notebook 08 output artifacts, where supplied.
7. Produce a reproducible research summary suitable for the final ECIPS report.

## Non-negotiable methodological safeguards

- The target is the dataset's **Kaggle-provided risk label**, not independently verified fraud ground truth.
- Preserve the previously audited source hash and conservative feature set.
- Exclude `risk_score`, `transaction_type`, `age_group`, `purchase_pattern`, sender/receiver addresses, and raw `timestamp` from predictors.
- Use `timestamp` only to order observations and define temporal windows.
- Do not use the final test partition to select models, calibrators, thresholds, or hyperparameters.
- Compare all candidate models under identical splits and metrics.
- A high-risk screening alert is not a fraud determination or an automatic blocking recommendation.
- Confidence intervals from an ordinary bootstrap describe resampling uncertainty under its assumptions; temporal dependence may make them optimistic. A moving-block bootstrap is also included as a sensitivity analysis.

In [ ]:
# 1. Setup
from pathlib import Path
import json, hashlib, warnings, importlib.util, sys, zipfile, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, average_precision_score, roc_auc_score,
    classification_report, confusion_matrix, brier_score_loss
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("/content/metaverse_transactions_dataset.csv")
OUTPUT_DIR = Path("/content/ecips_notebook09_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA256 = "c7eae68df7ff627089b44df6123c4974e71065f4c342b45a4bf4664bdea6beb0"
TARGET = "anomaly"
HIGH_LABEL = "high_risk"
FEATURES_NUMERIC = ["amount", "hour_of_day", "login_frequency", "session_duration", "ip_prefix"]
FEATURES_CATEGORICAL = ["location_region"]
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL

TRAIN_FRAC = 0.70
VAL_FRAC = 0.10
TEST_FRAC = 0.20

# Threshold selection policy is fixed before inspecting the final test partition.
MIN_HIGH_RISK_RECALL = 0.80
N_BOOTSTRAP = 1000
BLOCK_LENGTH = 250
N_TEMPORAL_FOLDS = 4

print("Python:", sys.version)
print("Output:", OUTPUT_DIR)

## 2. Load and verify the original dataset

Upload the same CSV used in Notebooks 01–08. This notebook stops if the SHA-256 differs from the audited source. Do not silently substitute a revised dataset.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Upload metaverse_transactions_dataset.csv to Colab or change DATA_PATH.")

df = pd.read_csv(DATA_PATH)
source_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Shape:", df.shape)
print("SHA-256:", source_hash)
if source_hash != EXPECTED_SHA256:
    raise ValueError("Dataset hash mismatch. Verify source/version before continuing.")

required = set(FEATURES + [TARGET, "timestamp"])
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
if df["timestamp"].isna().any():
    raise ValueError("Invalid timestamps found.")
df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

PROHIBITED = {
    "risk_score", "transaction_type", "age_group", "purchase_pattern",
    "sending_address", "receiving_address", "timestamp", TARGET
}
assert not (set(FEATURES) & PROHIBITED)

print("Rows/columns:", df.shape)
display(df[TARGET].value_counts().rename("count").to_frame())

## 3. Fixed chronological train / validation / test partitions

The primary split matches the Notebook 07–08 design: first 70% train, next 10% validation, final 20% test. All model comparisons and threshold selection use these same partitions.

In [ ]:
n = len(df)
train_end = int(n * TRAIN_FRAC)
val_end = int(n * (TRAIN_FRAC + VAL_FRAC))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]
LABELS = sorted(y_train.unique().tolist())

if set(y_val.unique()) - set(LABELS) or set(y_test.unique()) - set(LABELS):
    raise ValueError("Class absent from training but present later; inspect split design.")

split_summary = pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "start": [train_df.timestamp.min(), val_df.timestamp.min(), test_df.timestamp.min()],
    "end": [train_df.timestamp.max(), val_df.timestamp.max(), test_df.timestamp.max()]
})
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "09_split_summary.csv", index=False)

## 4. Candidate model construction

ExtraTrees and Logistic Regression are included directly. LightGBM is included if installed. All candidates use identical features, preprocessing, and chronological partitions. If LightGBM is missing, install it in the optional cell below and rerun from the top.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, FEATURES_NUMERIC),
    ("cat", categorical_pipe, FEATURES_CATEGORICAL)
])

def build_pipeline(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])

candidate_estimators = {
    "ExtraTrees_balanced": ExtraTreesClassifier(
        n_estimators=600, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "ExtraTrees_unweighted": ExtraTreesClassifier(
        n_estimators=600, min_samples_leaf=2, class_weight=None,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LogisticRegression_balanced": LogisticRegression(
        max_iter=4000, class_weight="balanced", C=1.0,
        random_state=RANDOM_STATE
    ),
    "RandomForest_balanced": RandomForestClassifier(
        n_estimators=600, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

if importlib.util.find_spec("lightgbm"):
    from lightgbm import LGBMClassifier
    candidate_estimators["LightGBM_balanced"] = LGBMClassifier(
        n_estimators=600, learning_rate=0.04, num_leaves=31,
        class_weight="balanced", random_state=RANDOM_STATE, verbosity=-1
    )
else:
    print("LightGBM is not installed; this candidate will be skipped.")

print("Candidates:", list(candidate_estimators))

In [ ]:
# Fit all candidates on training data; compare on the same validation partition.
fitted = {}
val_prob = {}
val_rows = []

for name, estimator in candidate_estimators.items():
    model = build_pipeline(estimator)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    proba = model.predict_proba(X_val)
    fitted[name] = model
    val_prob[name] = proba
    hi = list(model.classes_).index(HIGH_LABEL)
    y_hi = (y_val == HIGH_LABEL).astype(int)
    p_hi = (pred == HIGH_LABEL).astype(int)
    val_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, pred),
        "macro_f1": f1_score(y_val, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_val, pred, average="weighted", zero_division=0),
        "high_risk_precision_argmax": precision_score(y_hi, p_hi, zero_division=0),
        "high_risk_recall_argmax": recall_score(y_hi, p_hi, zero_division=0),
        "high_risk_ap": average_precision_score(y_hi, proba[:, hi]),
        "high_risk_roc_auc": roc_auc_score(y_hi, proba[:, hi])
    })

val_comparison = pd.DataFrame(val_rows).sort_values(
    ["macro_f1", "balanced_accuracy", "high_risk_recall_argmax"],
    ascending=False
).reset_index(drop=True)
display(val_comparison)
val_comparison.to_csv(OUTPUT_DIR / "09_validation_model_comparison.csv", index=False)

# Two objective-specific candidates are retained for analysis:
# A) multiclass champion by macro-F1
# B) high-risk probability champion by average precision
macro_champion_name = val_comparison.iloc[0]["model"]
screen_champion_name = val_comparison.sort_values(
    ["high_risk_ap", "high_risk_recall_argmax"], ascending=False
).iloc[0]["model"]

print("Macro-F1 champion:", macro_champion_name)
print("High-risk AP champion:", screen_champion_name)

## 5. Select a high-risk threshold separately for every candidate

For each candidate, find a validation threshold meeting the predeclared recall target (if feasible), then minimize validation false-negative/false-positive cost under an explicitly illustrative 10:1 cost ratio. This allows fairer comparison of ExtraTrees, LightGBM, and Logistic Regression as screening models.

The selected threshold for each candidate is frozen before test evaluation.

In [ ]:
FN_COST, FP_COST = 10.0, 1.0
threshold_grid = np.linspace(0.01, 0.99, 197)
threshold_rows = []
chosen_thresholds = {}

for name, model in fitted.items():
    classes = list(model.classes_)
    hi = classes.index(HIGH_LABEL)
    scores = val_prob[name][:, hi]
    truth = (y_val == HIGH_LABEL).astype(int).to_numpy()
    rows = []
    for threshold in threshold_grid:
        pred = (scores >= threshold).astype(int)
        tp = int(((pred == 1) & (truth == 1)).sum())
        fp = int(((pred == 1) & (truth == 0)).sum())
        fn = int(((pred == 0) & (truth == 1)).sum())
        tn = int(((pred == 0) & (truth == 0)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        cost = FN_COST*fn + FP_COST*fp
        rows.append({
            "model": name, "threshold": threshold, "tp": tp, "fp": fp,
            "fn": fn, "tn": tn, "precision": prec, "recall": rec,
            "f1": f1, "illustrative_cost": cost
        })
    curve = pd.DataFrame(rows)
    eligible = curve[curve.recall >= MIN_HIGH_RISK_RECALL]
    if len(eligible):
        chosen = eligible.sort_values(["illustrative_cost", "f1"], ascending=[True, False]).iloc[0]
        feasible = True
    else:
        chosen = curve.sort_values(["recall", "illustrative_cost"], ascending=[False, True]).iloc[0]
        feasible = False
    chosen_thresholds[name] = float(chosen.threshold)
    chosen_dict = chosen.to_dict()
    chosen_dict["recall_constraint_met"] = feasible
    threshold_rows.append(chosen_dict)

threshold_selection = pd.DataFrame(threshold_rows)
display(threshold_selection)
threshold_selection.to_csv(OUTPUT_DIR / "09_validation_threshold_selection.csv", index=False)

plt.figure(figsize=(9, 5))
for name, model in fitted.items():
    hi = list(model.classes_).index(HIGH_LABEL)
    truth = (y_val == HIGH_LABEL).astype(int)
    from sklearn.metrics import precision_recall_curve
    precision, recall, _ = precision_recall_curve(truth, val_prob[name][:, hi])
    plt.plot(recall, precision, label=name)
plt.xlabel("High-risk recall")
plt.ylabel("High-risk precision")
plt.title("Validation precision–recall comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Final test evaluation — models and thresholds now frozen

The test partition is evaluated once for all candidates. No test metric is used to revise model choice or thresholds. Report multiclass performance and the one-vs-rest high-risk screening results separately.

In [ ]:
test_rows = []
test_class_reports = {}
test_confusions = {}
test_screen_rows = []
test_screen_confusions = {}

for name, model in fitted.items():
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    classes = list(model.classes_)
    hi = classes.index(HIGH_LABEL)

    test_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0)
    })
    test_class_reports[name] = pd.DataFrame(
        classification_report(y_test, pred, labels=classes, output_dict=True, zero_division=0)
    ).T
    test_confusions[name] = pd.DataFrame(
        confusion_matrix(y_test, pred, labels=classes),
        index=[f"actual_{c}" for c in classes],
        columns=[f"predicted_{c}" for c in classes]
    )

    threshold = chosen_thresholds[name]
    truth = (y_test == HIGH_LABEL).astype(int).to_numpy()
    screen_pred = (proba[:, hi] >= threshold).astype(int)
    tp = int(((screen_pred == 1) & (truth == 1)).sum())
    fp = int(((screen_pred == 1) & (truth == 0)).sum())
    fn = int(((screen_pred == 0) & (truth == 1)).sum())
    tn = int(((screen_pred == 0) & (truth == 0)).sum())
    test_screen_rows.append({
        "model": name, "threshold_from_validation": threshold,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision_score(truth, screen_pred, zero_division=0),
        "recall": recall_score(truth, screen_pred, zero_division=0),
        "f1": f1_score(truth, screen_pred, zero_division=0),
        "average_precision": average_precision_score(truth, proba[:, hi]),
        "roc_auc": roc_auc_score(truth, proba[:, hi]),
        "alerts": int(screen_pred.sum()),
        "illustrative_cost": FN_COST*fn + FP_COST*fp
    })
    test_screen_confusions[name] = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["actual_not_high_risk", "actual_high_risk"],
        columns=["no_alert", "alert"]
    )

test_comparison = pd.DataFrame(test_rows).sort_values("macro_f1", ascending=False)
screen_comparison = pd.DataFrame(test_screen_rows).sort_values(
    ["average_precision", "recall"], ascending=False
)
display(test_comparison)
display(screen_comparison)
test_comparison.to_csv(OUTPUT_DIR / "09_final_test_multiclass_comparison.csv", index=False)
screen_comparison.to_csv(OUTPUT_DIR / "09_final_test_high_risk_screening_comparison.csv", index=False)

for name in fitted:
    test_class_reports[name].to_csv(OUTPUT_DIR / f"09_report_{name}.csv")
    test_confusions[name].to_csv(OUTPUT_DIR / f"09_confusion_{name}.csv")
    test_screen_confusions[name].to_csv(OUTPUT_DIR / f"09_screen_confusion_{name}.csv")

## 7. Probability calibration

Calibration is assessed on validation and test partitions for the selected multiclass champion and high-risk screening champion. Brier score and reliability diagrams help determine whether predicted probabilities correspond to observed label frequencies. Calibration on test is diagnostic only; do not use it to refit a calibrator or alter thresholds.

In [ ]:
calibration_rows = []
for name in sorted(set([macro_champion_name, screen_champion_name])):
    model = fitted[name]
    hi = list(model.classes_).index(HIGH_LABEL)
    val_scores = val_prob[name][:, hi]
    test_scores = model.predict_proba(X_test)[:, hi]
    yv = (y_val == HIGH_LABEL).astype(int)
    yt = (y_test == HIGH_LABEL).astype(int)
    calibration_rows.append({
        "model": name,
        "validation_brier": brier_score_loss(yv, val_scores),
        "test_brier": brier_score_loss(yt, test_scores)
    })

    plt.figure(figsize=(6, 5))
    for split_name, truth, scores in [
        ("Validation", yv, val_scores),
        ("Test", yt, test_scores)
    ]:
        frac_pos, mean_pred = calibration_curve(
            truth, scores, n_bins=10, strategy="quantile"
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=split_name)
    plt.plot([0,1], [0,1], linestyle="--", label="Perfect calibration")
    plt.xlabel("Mean predicted high-risk probability")
    plt.ylabel("Observed high-risk label frequency")
    plt.title(f"High-risk calibration — {name}")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

calibration_results = pd.DataFrame(calibration_rows)
display(calibration_results)
calibration_results.to_csv(OUTPUT_DIR / "09_calibration_metrics.csv", index=False)

## 8. Bootstrap confidence intervals on the final test partition

For each candidate's macro-F1 and high-risk screening precision/recall, compute percentile confidence intervals using:
- ordinary row bootstrap; and
- moving-block bootstrap, which samples contiguous blocks in chronological order.

These are descriptive uncertainty estimates. The block bootstrap is a sensitivity analysis; block length is a design choice and does not eliminate all dependence or distribution shift.

In [ ]:
def macro_f1_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def screening_metrics(y_true, y_score, threshold):
    truth = (np.asarray(y_true) == HIGH_LABEL).astype(int)
    pred = (np.asarray(y_score) >= threshold).astype(int)
    return np.array([
        f1_score(truth, pred, zero_division=0),
        precision_score(truth, pred, zero_division=0),
        recall_score(truth, pred, zero_division=0)
    ])

def bootstrap_indices(n, rng, block=False, block_length=250):
    if not block:
        return rng.integers(0, n, size=n)
    pieces = []
    while sum(len(x) for x in pieces) < n:
        start = int(rng.integers(0, max(1, n - block_length + 1)))
        pieces.append(np.arange(start, min(start + block_length, n)))
    return np.concatenate(pieces)[:n]

rng = np.random.default_rng(RANDOM_STATE)
bootstrap_rows = []

for name, model in fitted.items():
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    hi = list(model.classes_).index(HIGH_LABEL)
    threshold = chosen_thresholds[name]
    scores = proba[:, hi]

    for method, is_block in [("row_bootstrap", False), ("moving_block_bootstrap", True)]:
        values = []
        for _ in range(N_BOOTSTRAP):
            idx = bootstrap_indices(len(y_test), rng, block=is_block, block_length=BLOCK_LENGTH)
            yb = y_test.iloc[idx].to_numpy()
            pb = np.asarray(pred)[idx]
            sb = np.asarray(scores)[idx]
            mf1 = macro_f1_metric(yb, pb)
            scr = screening_metrics(yb, sb, threshold)
            values.append([mf1, *scr])
        arr = np.asarray(values)
        names = ["macro_f1", "high_risk_screen_f1", "high_risk_precision", "high_risk_recall"]
        for j, metric in enumerate(names):
            bootstrap_rows.append({
                "model": name, "bootstrap_method": method, "metric": metric,
                "estimate": float(np.mean(arr[:, j])),
                "ci_2_5": float(np.quantile(arr[:, j], 0.025)),
                "ci_97_5": float(np.quantile(arr[:, j], 0.975)),
                "bootstrap_replicates": N_BOOTSTRAP,
                "block_length": BLOCK_LENGTH if is_block else None
            })

bootstrap_results = pd.DataFrame(bootstrap_rows)
display(bootstrap_results)
bootstrap_results.to_csv(OUTPUT_DIR / "09_bootstrap_confidence_intervals.csv", index=False)

## 9. Multiple forward-chaining temporal evaluation windows

This section performs four expanding-window evaluations before the final 20% test partition. Each fold trains only on earlier observations and validates on the next chronological segment. It reports macro-F1, balanced accuracy, and high-risk average precision for each candidate.

These folds are a robustness analysis, not a substitute for external validation on a later independently collected dataset.

In [ ]:
# Create four sequential validation windows within the first 80% of the data.
development_end = val_end
dev = df.iloc[:development_end].copy()
fold_rows = []

# Each fold uses an expanding training prefix and the next contiguous window.
initial_train_end = int(len(dev) * 0.55)
remaining = len(dev) - initial_train_end
fold_size = max(1, remaining // N_TEMPORAL_FOLDS)

for fold in range(N_TEMPORAL_FOLDS):
    tr_end = initial_train_end + fold * fold_size
    va_end = min(len(dev), tr_end + fold_size)
    if va_end <= tr_end:
        continue
    fold_train = dev.iloc[:tr_end]
    fold_val = dev.iloc[tr_end:va_end]
    if fold_val[TARGET].nunique() < len(LABELS):
        print(f"Fold {fold+1}: skipped — validation window lacks at least one class.")
        continue

    for name, estimator in candidate_estimators.items():
        model = build_pipeline(estimator)
        model.fit(fold_train[FEATURES], fold_train[TARGET])
        pred = model.predict(fold_val[FEATURES])
        proba = model.predict_proba(fold_val[FEATURES])
        hi = list(model.classes_).index(HIGH_LABEL)
        yhi = (fold_val[TARGET] == HIGH_LABEL).astype(int)
        fold_rows.append({
            "fold": fold + 1, "model": name,
            "train_rows": len(fold_train), "validation_rows": len(fold_val),
            "train_end": str(fold_train.timestamp.max()),
            "validation_start": str(fold_val.timestamp.min()),
            "validation_end": str(fold_val.timestamp.max()),
            "macro_f1": f1_score(fold_val[TARGET], pred, average="macro", zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(fold_val[TARGET], pred),
            "high_risk_average_precision": average_precision_score(yhi, proba[:, hi])
        })

temporal_results = pd.DataFrame(fold_rows)
display(temporal_results)
temporal_results.to_csv(OUTPUT_DIR / "09_temporal_robustness_folds.csv", index=False)

if len(temporal_results):
    temporal_summary = temporal_results.groupby("model").agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        high_risk_ap_mean=("high_risk_average_precision", "mean"),
        high_risk_ap_std=("high_risk_average_precision", "std")
    ).reset_index()
    display(temporal_summary)
    temporal_summary.to_csv(OUTPUT_DIR / "09_temporal_robustness_summary.csv", index=False)

## 10. Consolidate prior Notebook 07 and 08 results

Upload the relevant prior output CSV/JSON files to the same Colab session. This cell reads them when present and creates a compact cross-notebook record. Missing optional files are listed rather than inferred or fabricated.

Recommended files:
- `07_final_test_summary.csv`
- `07_final_test_classification_report.csv`
- `07_high_risk_test_threshold_metrics.csv`
- `07_validation_model_comparison.csv`
- `08_final_test_summary.csv`
- `08_final_test_high_risk_screening_metrics.csv`
- `08_validation_model_comparison.csv`
- `08_reproducibility_metadata.json`

In [ ]:
PRIOR_FILES = [
    "07_final_test_summary.csv",
    "07_final_test_classification_report.csv",
    "07_high_risk_test_threshold_metrics.csv",
    "07_validation_model_comparison.csv",
    "08_final_test_summary.csv",
    "08_final_test_high_risk_screening_metrics.csv",
    "08_validation_model_comparison.csv",
    "08_reproducibility_metadata.json"
]
prior_inventory = []
for filename in PRIOR_FILES:
    path = Path("/content") / filename
    prior_inventory.append({
        "file": filename,
        "available": path.exists(),
        "bytes": path.stat().st_size if path.exists() else None
    })
display(pd.DataFrame(prior_inventory))
pd.DataFrame(prior_inventory).to_csv(OUTPUT_DIR / "09_prior_artifact_inventory.csv", index=False)

prior_summaries = {}
for filename in PRIOR_FILES:
    path = Path("/content") / filename
    if path.exists():
        try:
            if filename.endswith(".json"):
                prior_summaries[filename] = json.loads(path.read_text())
            else:
                prior_summaries[filename] = pd.read_csv(path).to_dict(orient="records")
        except Exception as exc:
            prior_summaries[filename] = {"read_error": str(exc)}

with open(OUTPUT_DIR / "09_prior_notebook_results_consolidated.json", "w") as f:
    json.dump(prior_summaries, f, indent=2, default=str)

## 11. Research findings and limitations register

This cell creates a structured checklist. Complete the interpretation fields after inspecting outputs. Do not convert model metrics into claims of verified fraud detection.

In [ ]:
research_register = pd.DataFrame([
    ["Target validity", "Kaggle-provided anomaly/risk labels", "Not independently verified fraud outcomes", "Critical limitation"],
    ["Leakage control", "Known proxy-like fields excluded", "Independence from label-generation process not established", "Continue provenance investigation"],
    ["Model selection", "Validation macro-F1 primary; balanced accuracy and high-risk recall secondary", "Different objective may select a different model", "Report objective-specific champions"],
    ["High-risk screening", "Validation-only thresholds per candidate", "Recall/precision trade-off and alert burden must be disclosed", "No automatic blocking"],
    ["Temporal robustness", "Expanding-window forward validation", "One source dataset and time period only", "Seek later external data"],
    ["Uncertainty", "Row and moving-block bootstrap intervals", "Intervals depend on resampling assumptions", "Interpret cautiously"],
    ["Calibration", "Brier scores and reliability plots", "Calibration can drift over time", "Monitor on future labelled data"],
    ["Explainability", "Prior permutation importance; this notebook prioritizes comparable metrics", "Importance is not causal or provenance evidence", "Avoid causal language"]
], columns=["topic", "evidence_or_method", "limitation", "research_implication"])
display(research_register)
research_register.to_csv(OUTPUT_DIR / "09_research_limitations_register.csv", index=False)

## 12. Reproducibility metadata and package outputs

In [ ]:
metadata = {
    "notebook": "09_Final_Model_Validation_Statistical_Robustness_and_Research_Consolidation",
    "source_file": DATA_PATH.name,
    "source_sha256": source_hash,
    "dataset_shape": list(df.shape),
    "target": TARGET,
    "target_interpretation": "Kaggle-provided risk label; not independently verified fraud ground truth",
    "features": FEATURES,
    "excluded_predictors": sorted(PROHIBITED),
    "split": {
        "method": "chronological 70/10/20",
        "train_end": str(train_df.timestamp.max()),
        "validation_end": str(val_df.timestamp.max()),
        "test_start": str(test_df.timestamp.min())
    },
    "candidate_models": list(candidate_estimators.keys()),
    "validation_macro_f1_champion": macro_champion_name,
    "validation_high_risk_ap_champion": screen_champion_name,
    "thresholds_selected_on_validation_only": chosen_thresholds,
    "minimum_validation_high_risk_recall": MIN_HIGH_RISK_RECALL,
    "illustrative_cost_ratio_fn_to_fp": FN_COST / FP_COST,
    "bootstrap_replicates": N_BOOTSTRAP,
    "moving_block_length_rows": BLOCK_LENGTH,
    "temporal_folds_requested": N_TEMPORAL_FOLDS,
    "limitations": [
        "Risk labels are not independently verified fraud outcomes.",
        "Feature independence from label generation is not established.",
        "A single chronological holdout does not establish deployment performance.",
        "Bootstrap intervals depend on resampling assumptions and may not capture all temporal dependence.",
        "Threshold costs are illustrative and require stakeholder validation.",
        "No automatic transaction blocking is recommended."
    ]
}
with open(OUTPUT_DIR / "09_reproducibility_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# Create a human-readable summary of the principal comparisons.
summary_lines = [
    "# Notebook 09 — Results Summary",
    "",
    f"- Dataset rows: {len(df):,}; columns: {df.shape[1]}",
    f"- Source SHA-256: `{source_hash}`",
    f"- Validation macro-F1 champion: **{macro_champion_name}**",
    f"- Validation high-risk AP champion: **{screen_champion_name}**",
    "",
    "## Final test multiclass comparison",
    test_comparison.to_markdown(index=False),
    "",
    "## Final test high-risk screening comparison",
    screen_comparison.to_markdown(index=False),
    "",
    "## Interpretation guardrail",
    "The target is the Kaggle-provided risk label, not independently verified fraud ground truth."
]
(OUTPUT_DIR / "09_results_summary.md").write_text("\n".join(summary_lines))

zip_path = Path("/content/09_notebook_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.iterdir():
        z.write(p, arcname=f"ecips_notebook09_outputs/{p.name}")

print("Outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name, p.stat().st_size, "bytes")
print("ZIP created:", zip_path)

## 13. Final reporting checklist

Before finalizing the ECIPS research report, answer:

1. Which model is the macro-F1 champion on validation, and does it remain competitive on test?
2. Which model is the high-risk average-precision champion?
3. At each model's validation-selected threshold, how many high-risk labels are detected, missed, and falsely alerted on test?
4. How do row-bootstrap and moving-block-bootstrap confidence intervals differ?
5. Are model rankings stable across forward-chaining windows?
6. Which models have better probability calibration?
7. What operational alert volume would the chosen threshold create?
8. Are the assumed cost weights approved by domain stakeholders?
9. What independent evidence is available about the label-generation process?
10. What future dataset or prospective evaluation is needed?

**Final research position:** select a model only relative to a clearly stated objective. Do not claim that a classifier detects real-world fraud unless independently verified outcomes and a defensible evaluation design support that claim.